# 阅读与修改 PyTorch 项目代码

## 学习目标

从训练入口追踪数据、模型、loss、优化器和 checkpoint，使用签名与源码定位职责，并完成一个范围明确的模型修改。

## 概念模型

阅读项目时先找执行入口，再沿调用链确认输入输出契约。不要从目录中的每个文件顺序阅读；围绕一次训练的数据流建立地图。

In [ ]:
import inspect
from pathlib import Path
import torch
from torch import nn
from common import engine, checkpoint, models, runtime

root = Path.cwd()
print('course root:', root)
print('train signature:', inspect.signature(engine.train_one_epoch))
print('evaluate signature:', inspect.signature(engine.evaluate))
print('checkpoint signature:', inspect.signature(checkpoint.save_checkpoint))

### 实验 1：建立训练数据流地图

一次分类训练可以压缩为：Dataset/DataLoader -> model -> logits -> loss -> backward -> optimizer -> validation -> checkpoint。

In [ ]:
source = inspect.getsource(engine._run_epoch)
required_steps = ['inputs.to(device)', 'optimizer.zero_grad', 'loss.backward', 'optimizer.step']
for step in required_steps:
    assert step in source, step
print('engine contract:', required_steps)
print('returns:', inspect.signature(engine.EpochResult))

### 实验 2：理解模型边界

模型负责从输入 Tensor 产生 logits；训练 engine 不需要知道内部使用 CNN、RNN 还是 Transformer。

In [ ]:
base = models.ImageClassifier(channels=1, num_classes=10)
sample = torch.randn(2, 1, 28, 28)
assert base(sample).shape == (2, 10)
print('feature modules:', list(base.features.named_children()))
print('classifier:', base.classifier)

### 实验 3：做一个局部修改并保护契约

下面只替换分类头并增加 Dropout，保持输入格式和 logits shape 不变。这种修改不需要改动训练 engine。

In [ ]:
modified = models.ImageClassifier(channels=1, num_classes=10, dropout=0.3)
modified.classifier[-1] = nn.Linear(32 * 4 * 4, 5)
output = modified(sample)
assert output.shape == (2, 5)
assert any(isinstance(module, nn.Dropout) for module in modified.modules())
print('modified logits:', output.shape, 'parameters:', sum(p.numel() for p in modified.parameters()))

## 检查点

指出训练入口、数据入口、模型边界、loss 和 checkpoint 分别位于哪里；解释为什么修改分类头不需要修改通用训练循环。

## 试一试

阅读 `examples/train_image_classifier.py`，画出它与 `common` 模块的调用关系；再增加一个模型参数并确认 CLI、checkpoint 和 shape 测试是否需要变化。

## 常见错误与调试

从所有文件第一行开始顺序阅读、只看类名不检查输入输出、修改共享 engine 解决模型内部问题、没有测试修改后的 shape、忽略 checkpoint 与旧模型结构的兼容性。